## Start Spark and set the path.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

PROJECT_ROOT = Path.cwd().parent
SILVER_DIR = (PROJECT_ROOT / "silver").as_posix() # read from silver as parquet// a type of file format that is optimized for big data processing

builder = (
    SparkSession.builder
    .appName("movielens-part3")
    .master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 3.5.8


## Open the silver ratings table and show its history
- `DeltaTable.forPath(spark, ...)` opens a handle to the table. It is not a DataFrame. A DataFrame is just rows, while a DeltaTable also gives access to Delta-only commands: history, merge, restore, delete. Nothing is read into memory yet.
- `.history()` reads the `_delta_log` folder and returns a DataFrame with one row per commit. It's the same log we inspected by hand in Part 2, now readable through Spark.
- `.select(...)` keeps four things: which version, when, what kind of operation, and how many rows that commit wrote. The last one uses `operationMetrics`, a column holding a small dictionary of stats. ["numOutputRows"] pulls out one entry and `.alias(...)` renames it.

In [ ]:
from delta.tables import DeltaTable

silver_ratings_table = DeltaTable.forPath(spark, f"{SILVER_DIR}/silver_ratings") # opening the silver ratings

# show the history of the silver ratings table
silver_ratings_table.history().select(
    "version",
    "timestamp",
    "operation",
    F.col("operationMetrics")["numOutputRows"].alias("rows_written")
).show(truncate=False)

+-------+-----------------------+---------+------------+
|version|timestamp              |operation|rows_written|
+-------+-----------------------+---------+------------+
|4      |2026-09-19 15:46:12.29 |WRITE    |100838      |
|3      |2026-09-19 15:46:00.566|WRITE    |100838      |
|2      |2026-09-19 15:45:48.155|WRITE    |100836      |
|1      |2026-09-18 00:47:05.892|WRITE    |100838      |
|0      |2026-09-18 00:42:02.686|WRITE    |100836      |
+-------+-----------------------+---------+------------+



## Try a bad write and watch Delta refuse it.
build one row that matches the table's columns, plus one extra column the table doesn't have (`mood`). Then try to append it to `silver_ratings`. Delta should reject it, and the table should stay exactly as it was.

- `path = ...` is only a shortcut so we don't retype the table location.
- `rows_before` and `versions_before` record the table's state before the attempt: how many rows, and how many commits in its log. Afterwards we compare, so we prove nothing changed instead of assuming it.
- `bad_batch = (...)` builds the 1-row test batch from `spark.range(1)`, the same trick as Part 2. Every column matches the table, with the right types (the `.cast("int")` calls are there for that reason), plus the one extra column, `mood`. `.drop("id")` removes the helper column that `spark.range` creates.
- `try: ... except` Exception as `e`: is new Python. It means "attempt the code under try. If it crashes, don't stop the notebook. Run the except block instead and hand me the error as e." Without it, the failure would fill the screen with a huge traceback. Here we expect a failure and only want to read its message.
- `bad_batch.write...mode("append").save(path)` is the write we expect to fail. It's the same append pattern as Bronze in Part 1.
`type(e).__name__` prints the kind of error. `str(e)[:400]` prints only the first 400 characters of the message, so the output stays readable.
- The last block re-reads the row count and version count and prints before and after for each.

In [3]:
path = f"{SILVER_DIR}/silver_ratings"

rows_before = spark.read.format("delta").load(path).count()
versions_before = silver_ratings_table.history().count()

bad_batch = (
    spark.range(1)
    .withColumn("userId", F.lit(1).cast("int"))
    .withColumn("movieId", F.lit(1).cast("int"))
    .withColumn("rating", F.lit(4.0))
    .withColumn("rated_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("bad_test"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("bad_test"))
    .withColumn("mood", F.lit("happy"))
    .drop("id")
)

try:
    bad_batch.write.format("delta").mode("append").save(path)
    print("write was ACCEPTED")
except Exception as e:
    print("write REJECTED:", type(e).__name__)
    print(str(e)[:400])

rows_after = spark.read.format("delta").load(path).count()
versions_after = silver_ratings_table.history().count()
print("rows:", rows_before, "->", rows_after)
print("versions:", versions_before, "->", versions_after)

write REJECTED: AnalysisException
[_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: c0ec2770-21bc-4a08-819a-359e6587857f).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific t
rows: 100838 -> 100838
versions: 5 -> 5


## Check the existing data, then add the rule
**A CHECK constraint** is a rule stored on the table itself. Delta tests every future write against it. Before adding one, we have to confirm the rows already in the table obey it. Otherwise Delta refuses to add the rule.

- `violations = (...)` block reads the table and keeps only rows that would break the rule: rating below 0.5, above 5.0, or missing entirely. The | means "or". Then it counts them. This is the "clean existing data first" check.
- `spark.sql("ALTER TABLE ... ADD CONSTRAINT ...")` is a Spark SQL command, the same kind of SQL you'd use in a database. It attaches a rule named `rating_range` to the table: `rating >= 0.5 AND rating <= 5.0`. The delta. followed by the path in backticks tells SQL "the table stored at this folder", since our table has no registered name.
- The history line shows the two newest commits. `limit(2)` keeps just the top two, since history lists newest first.
- `.detail()` returns the table's metadata, and properties is the setting where Delta stores constraints.

In [4]:
violations = (
    spark.read.format("delta").load(path)
    .filter((F.col("rating") < 0.5) | (F.col("rating") > 5.0) | F.col("rating").isNull())
    .count()
)
print("existing rows breaking the rule:", violations)

spark.sql(f"ALTER TABLE delta.`{path}` ADD CONSTRAINT rating_range CHECK (rating >= 0.5 AND rating <= 5.0)")

silver_ratings_table.history().select("version", "operation").limit(2).show(truncate=False)
silver_ratings_table.detail().select("properties").show(truncate=False)

existing rows breaking the rule: 0
+-------+--------------+
|version|operation     |
+-------+--------------+
|5      |ADD CONSTRAINT|
|4      |WRITE         |
+-------+--------------+

+-------------------------------------------------------------------+
|properties                                                         |
+-------------------------------------------------------------------+
|{delta.constraints.rating_range -> rating >= 0.5 AND rating <= 5.0}|
+-------------------------------------------------------------------+



## Test the checker

In [5]:
rows_before = spark.read.format("delta").load(path).count()
versions_before = silver_ratings_table.history().count()

bad_rating = (
    spark.range(1)
    .withColumn("userId", F.lit(9998).cast("int"))
    .withColumn("movieId", F.lit(1).cast("int"))
    .withColumn("rating", F.lit(7.0))                  # intentionally invalid rating to break the constraint
    .withColumn("rated_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("bad_test"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("bad_test"))
    .drop("id")
)

try:
    bad_rating.write.format("delta").mode("append").save(path)
    print("write was ACCEPTED")
except Exception as e:
    print("write REJECTED:", type(e).__name__)
    print(str(e)[:500])

rows_after = spark.read.format("delta").load(path).count()
versions_after = silver_ratings_table.history().count()
print("rows:", rows_before, "->", rows_after)
print("versions:", versions_before, "->", versions_after)

write REJECTED: Py4JJavaError
An error occurred while calling o137.save.
: org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint rating_range ((rating >= 0.5BD) AND (rating <= 5.0BD)) violated by row with values:
 - rating : 7.0
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:78)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply
rows: 100838 -> 100838
versions: 6 -> 6


## Build the incoming batch and look at what it will collide with
**A MERGE** compares two things. The target is the table (silver_ratings). The source is a small batch of incoming rows. For each incoming row, MERGE looks for a row in the target with the same key, (userId, movieId). If it finds one, it can update it. If it finds none, it can insert the row. That's what "upsert" means: update + insert in one operation.

We'll send 3 incoming rows, each testing a different case:
| Incoming row | What it is | What should happen |
|---|---|---|
| user 1, movie 1, rating 3.0, dated now | A real correction (newer than the existing rating) | Update the existing row |
| user 1, movie 3, rating 1.0, dated 1999 | A stale rating (older than the existing one) | Ignore it |
| user 9999, movie 3, rating 2.5, dated now | A pair the table has never seen | Insert it |

- `incoming = (...)` builds the 3 rows from `spark.range(3)`, which gives ids 0, 1 and 2. 
- `F.when(condition, value).otherwise(other)` is Spark's if/else, and here it picks a different value per row based on id. Row 0 is the correction, row 1 the stale one, and row 2 the new pair. 
- `F.to_timestamp(F.lit("1999-01-01 00:00:00"))` turns that text into a real timestamp.

In [6]:
incoming = (
    spark.range(3)
    .withColumn("userId", F.when(F.col("id") == 2, 9999).otherwise(1).cast("int"))
    .withColumn("movieId", F.when(F.col("id") == 0, 1).otherwise(3).cast("int"))
    .withColumn("rating", F.when(F.col("id") == 0, 3.0).when(F.col("id") == 1, 1.0).otherwise(2.5))
    .withColumn("rated_at", F.when(F.col("id") == 1, F.to_timestamp(F.lit("1999-01-01 00:00:00"))).otherwise(F.current_timestamp()))
    .withColumn("batch_id", F.lit("merge_batch"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("merge_test"))
    .drop("id")
)

print("--- incoming batch ---")
incoming.select("userId", "movieId", "rating", "rated_at").show(truncate=False)

print("--- what the table holds for those same keys right now ---")
(
    spark.read.format("delta").load(path)
    .join(incoming.select("userId", "movieId"), on=["userId", "movieId"], how="inner")
    .select("userId", "movieId", "rating", "rated_at")
    .show(truncate=False)
)

--- incoming batch ---
+------+-------+------+--------------------------+
|userId|movieId|rating|rated_at                  |
+------+-------+------+--------------------------+
|1     |1      |3.0   |2026-09-19 22:15:09.286326|
|1     |3      |1.0   |1999-01-01 00:00:00       |
|9999  |3      |2.5   |2026-09-19 22:15:09.286326|
+------+-------+------+--------------------------+

--- what the table holds for those same keys right now ---
+------+-------+------+-------------------+
|userId|movieId|rating|rated_at           |
+------+-------+------+-------------------+
|1     |1      |4.0   |2000-07-30 21:45:03|
|1     |3      |4.0   |2000-07-30 21:20:47|
+------+-------+------+-------------------+



## Run the MERGE
- `rows_before` records the row count so we can compare afterwards.
- `silver_ratings_table.alias("t")` gives the target table the short name `t`.
- `.merge(incoming.alias("s"), "t.userId = s.userId AND t.movieId = s.movieId")` names the source s and states the matching rule. Two rows are "the same" when both `userId` and `movieId` are equal. 
- `.whenMatchedUpdateAll(condition="s.rated_at > t.rated_at")` says: when a match is found, overwrite all columns of the table's row with the incoming row's values, but only if the incoming `rated_at is` newer. That condition is the precedence rule. Without it, MERGE would let any incoming row overwrite the table, including our stale 1999 rating.
- `.whenNotMatchedInsertAll()` says: when no match is found, insert the incoming row as a new row.
- `.execute()` runs it. Nothing above this line changes the table. It just describes the MERGE, like the `.save()` trigger in Part 1.

In [7]:
rows_before = spark.read.format("delta").load(path).count()

(
    silver_ratings_table.alias("t")
    .merge(incoming.alias("s"), "t.userId = s.userId AND t.movieId = s.movieId")
    .whenMatchedUpdateAll(condition="s.rated_at > t.rated_at")
    .whenNotMatchedInsertAll()
    .execute()
)

rows_after = spark.read.format("delta").load(path).count()
print("rows:", rows_before, "->", rows_after)

print("--- the same keys after the MERGE ---")
(
    spark.read.format("delta").load(path)
    .join(incoming.select("userId", "movieId"), on=["userId", "movieId"], how="inner")
    .select("userId", "movieId", "rating", "rated_at")
    .orderBy("userId", "movieId")
    .show(truncate=False)
)

silver_ratings_table.history().select(
    "version",
    "operation",
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"),
    F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
).limit(2).show(truncate=False)

rows: 100838 -> 100839
--- the same keys after the MERGE ---
+------+-------+------+--------------------------+
|userId|movieId|rating|rated_at                  |
+------+-------+------+--------------------------+
|1     |1      |3.0   |2026-09-19 22:19:05.358022|
|1     |3      |4.0   |2000-07-30 21:20:47       |
|9999  |3      |2.5   |2026-09-19 22:19:05.358022|
+------+-------+------+--------------------------+

+-------+--------------+-------+--------+
|version|operation     |updated|inserted|
+-------+--------------+-------+--------+
|6      |MERGE         |1      |1       |
|5      |ADD CONSTRAINT|NULL   |NULL    |
+-------+--------------+-------+--------+



The MERGE did what we designed, on all three cases:

- **Update**: user 1 movie 1 went from 4.0 to 3.0, because the incoming row was newer.
- **Ignore**: user 1 movie 3 stayed at 4.0 (dated 2000). The stale 1999 rating arrived later than the existing row but was older by `rated_at`, and it lost. That is the check Day 2 couldn't make, because there the newest rating was also the last to arrive.
- **Insert**: user 9999 movie 3 was added, taking the table from 100,838 to 100,839 rows.
- **History**: version 6 is a MERGE with `updated = 1`, `inserted = 1`. The `NULL` on version 5 is normal, since adding a constraint writes no rows.

The incoming batch showed `rated_at = 22:15:09`, but the rows that landed have `22:19:05`. This is because `F.current_timestamp()` is recalculated every time the batch is used, so incoming is not a fixed batch. That would break the "run it twice" test planned. On a rerun, "now" would be even newer, so the MERGE would update those rows again. The fix is to build the batch with fixed timestamps.

In [8]:
FIXED_TS = "2026-09-19 22:30:00"

incoming_fixed = (
    spark.range(3)
    .withColumn("userId", F.when(F.col("id") == 2, 9999).otherwise(1).cast("int"))
    .withColumn("movieId", F.when(F.col("id") == 0, 1).otherwise(3).cast("int"))
    .withColumn("rating", F.when(F.col("id") == 0, 3.0).when(F.col("id") == 1, 1.0).otherwise(2.5))
    .withColumn("rated_at", F.when(F.col("id") == 1, F.to_timestamp(F.lit("1999-01-01 00:00:00"))).otherwise(F.to_timestamp(F.lit(FIXED_TS))))
    .withColumn("batch_id", F.lit("merge_batch"))
    .withColumn("ingested_at", F.to_timestamp(F.lit(FIXED_TS)))
    .withColumn("source_file", F.lit("merge_test"))
    .drop("id")
)

def apply_batch(batch):
    (
        silver_ratings_table.alias("t")
        .merge(batch.alias("s"), "t.userId = s.userId AND t.movieId = s.movieId")
        .whenMatchedUpdateAll(condition="s.rated_at > t.rated_at")
        .whenNotMatchedInsertAll()
        .execute()
    )

apply_batch(incoming_fixed)
apply_batch(incoming_fixed)

print("rows now:", spark.read.format("delta").load(path).count())

silver_ratings_table.history().select(
    "version",
    "operation",
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"),
    F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
).limit(3).show(truncate=False)

rows now: 100839
+-------+---------+-------+--------+
|version|operation|updated|inserted|
+-------+---------+-------+--------+
|8      |MERGE    |0      |0       |
|7      |MERGE    |2      |0       |
|6      |MERGE    |1      |1       |
+-------+---------+-------+--------+

